In [1]:
# !pip install aif360

In [2]:
import torch
from FairReg.Regularization.EqualizedOddsLoss import EqualizedOddsLoss
from Models.logistic_regression_net import LinearClassificationNet
import random
import numpy as np
import os
from utils.model_utils import ModelUtils
from FairReg.DPLUtils.regularization_config import RegularizationConfig
from FairReg.Learning.learning_new import Learning

from utils.dataset_utils import DatasetUtils
from utils.tabular_datasets_utils import load_dutch
from scipy.io import arff
from sklearn.model_selection import train_test_split
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import BinaryLabelDatasetMetric, ClassificationMetric
import pandas as pd
from sklearn.linear_model import LogisticRegression
from utils.tabular_datasets_utils import dataset_to_numpy, load_dutch

2024-03-11 09:11:49.891620: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-03-11 09:11:49.930915: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-03-11 09:11:49.930941: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-03-11 09:11:49.932339: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-03-11 09:11:49.939692: I tensorflow/core/platform/cpu_feature_guar

In [3]:
# ignore warnings

import warnings

warnings.filterwarnings("ignore")

In [4]:
class SPATIALFairnessModule:
    ## Assumes binary values (1,0) in all the inputs, we can add extra parameters if that’s not the case
    def __init__(self, _y_true, _y_pred, _groups, _group_name):
        self.y_true = _y_true  # Binary array. Contains the true label for each sample
        self.y_pred = _y_pred  # Binary array. Contains the predictions for each sample
        self.groups = _groups  # Binary array. Defines the demogaphic group membership of each sample
        self.grouping_name = _group_name  # String. Gives a name to the grouping. Example: Gender
        _label_names = ["Y"]
        _protected_attribute_names = [_group_name]
        _favorable_label = 1
        _unfavorable_label = 0
        _df = pd.DataFrame(columns=["Y", _group_name])
        _df["Y"] = _y_true
        _df[_group_name] = _groups
        self.aif_input_data = BinaryLabelDataset(
            df=_df,
            label_names=_label_names,
            protected_attribute_names=_protected_attribute_names,
            favorable_label=_favorable_label,
            unfavorable_label=_unfavorable_label,
        )
        self.privileged_groups = [{_group_name: 1}]
        self.unprivileged_groups = [{_group_name: 0}]
        self.input_metrics = BinaryLabelDatasetMetric(
            self.aif_input_data,
            unprivileged_groups=self.unprivileged_groups,
            privileged_groups=self.privileged_groups,
        )
        self.pred_aif_data = self.aif_input_data.copy(deepcopy=True)
        self.pred_aif_data.labels = _y_pred
        self.clf_metrics = ClassificationMetric(
            self.aif_input_data,
            self.pred_aif_data,
            unprivileged_groups=self.unprivileged_groups,
            privileged_groups=self.privileged_groups,
        )

# Train a model with Logistic Regression

In [5]:
# tmp = load_dutch(dataset_path="../dataset/dutch/")
# tmp = dataset_to_numpy(*tmp, num_sensitive_features=1)

# x = tmp[0]
# y = tmp[2]
# z = tmp[1]

# xyz = list(zip(x, y, z))
# random.shuffle(xyz)
# x, y, z = zip(*xyz)
# train_size = int(len(y) * 0.8)

# x_train = np.array(x[:train_size])
# x_test = np.array(x[train_size:])
# y_train = np.array(y[:train_size])
# y_test = np.array(y[train_size:])
# z_train = np.array(z[:train_size])
# z_test = np.array(z[train_size:])

In [6]:
# # Study how the dataset is distributed among the 4 possible combinations
# distributions = {}

# for index, (label, group) in enumerate(zip(y_train, z_train)):
#     if (label, group) not in distributions:
#         distributions[(label, group)] = []

#     distributions[(label, group)].append(index)

In [7]:
# for key in distributions:
#     print(f"Label: {key[0]}, Group: {key[1]}, Count: {len(distributions[key])}")

In [8]:
# index_to_be_removed = distributions[(0, 0)][0:7000]

In [9]:
# # remove indexes from x_train, y_train, z_train
# x_train = np.delete(x_train, index_to_be_removed, axis=0)
# y_train = np.delete(y_train, index_to_be_removed, axis=0)
# z_train = np.delete(z_train, index_to_be_removed, axis=0)


In [10]:
# clf = LogisticRegression(random_state=23252323).fit(x_train, y_train)

In [11]:
# y_predicted = clf.predict(x_test)
# y_predicted

In [12]:
# clf.score(x_test, y_test)

In [13]:
# fairness_module = SPATIALFairnessModule(_y_true=y_test, _y_pred=y_predicted, _groups=z_test, _group_name="Gender")

In [14]:
# print("average_abs_odds_difference: ", fairness_module.clf_metrics.average_abs_odds_difference())
# print("average_odds_difference: ", fairness_module.clf_metrics.average_odds_difference())
# print("average_predictive_value_difference: ", fairness_module.clf_metrics.average_predictive_value_difference())

# print("equalized_odds_difference: ", fairness_module.clf_metrics.equalized_odds_difference())
# print("error_rate_difference: ", fairness_module.clf_metrics.error_rate_difference())
# print("true_positive_rate_difference: ", fairness_module.clf_metrics.true_positive_rate_difference())

# Dutch Dataset

In [15]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
batch_size = 333

In [16]:
train_ds, test_ds = DatasetUtils.download_dataset(dataset_name="dutch", base_path="../dataset/dutch/")

Using ['sex_binary'] as sensitive feature(s).


In [17]:
train_loader = torch.utils.data.DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
)

test_loader = torch.utils.data.DataLoader(
    test_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
)

# Define a custom average predictive value difference

In [ ]:
# def count_true_positive(y_true, y_pred, group, sensitive_value):
#     true_positive = 0
#     for i in range(len(y_true)):
#         if y_true[i] == 1 and y_pred[i] == 1 and group[i] == sensitive_value:
#             true_positive += 1
#     return true_positive

# def count_false_positive(y_true, y_pred, group, sensitive_value):
#     false_positive = 0
#     for i in range(len(y_true)):
#         if y_true[i] == 0 and y_pred[i] == 1 and group[i] == sensitive_value:
#             false_positive += 1
#     return false_positive

# def count_true_negative(y_true, y_pred, group, sensitive_value):
#     true_negative = 0
#     for i in range(len(y_true)):
#         if y_true[i] == 0 and y_pred[i] == 0 and group[i] == sensitive_value:
#             true_negative += 1
#     return true_negative

# def count_false_negative(y_true, y_pred, group, sensitive_value):
#     false_negative = 0
#     for i in range(len(y_true)):
#         if y_true[i] == 1 and y_pred[i] == 0 and group[i] == sensitive_value:
#             false_negative += 1
#     return false_negative

# def compute_PPV(y_true, y_pred, group, sensitive_value):
#     true_positive = count_true_positive(y_true, y_pred, group, sensitive_value)
#     false_positive = count_false_positive(y_true, y_pred, group, sensitive_value)
#     return true_positive / (true_positive + false_positive)

# def compute_for(y_true, y_pred, group, sensitive_value):
#     true_negative = count_true_negative(y_true, y_pred, group, sensitive_value)
#     false_negative = count_false_negative(y_true, y_pred, group, sensitive_value)
#     return false_negative / (true_negative + false_negative)

In [ ]:
# def average_predictive_value_difference(y_true, y_pred, group, possible_sensitive_values):
#     max_average_predictive_value_difference = 0
#     for sensitive_value in possible_sensitive_values:
#         opposite_sensitive_value = abs(1 - sensitive_value)
#         current_average_predictive_value_difference = 0.5 * ((compute_PPV(y_true, y_pred, group, sensitive_value) - compute_PPV(y_true, y_pred, group, opposite_sensitive_value)) + (compute_for(y_true, y_pred, group, sensitive_value) - compute_for(y_true, y_pred, group, opposite_sensitive_value)))
#         max_average_predictive_value_difference = max(max_average_predictive_value_difference, current_average_predictive_value_difference)
#     return max_average_predictive_value_difference


# Train a model without Fairness Mitigation

In [18]:
# Create the model that we will train, for Dutch we will use a LinearClassificationNet
# defined inside this Library in the Models/logistic_regression_net.py file
model = LinearClassificationNet()
lr = 0.019925917176300392
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
epochs = 2
seed = 42

# seed the model
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)
torch.cuda.manual_seed_all(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
os.environ["PYTHONHASHSEED"] = str(seed)

In [19]:
# We don't want to use privacy in this case but we make the model private using
# the noise=0 because the code for learning the model only accept private models.
(
    private_model,
    private_optimizer,
    private_train_loader,
) = ModelUtils.create_private_model(
    model=model,
    epsilon=None,
    noise_multiplier=0,
    original_optimizer=optimizer,
    train_loader=train_loader,
    epochs=epochs,
    delta=0,
    MAX_GRAD_NORM=10000000000,  # since we just need to wrap the model without using privacy we use a high value here
    batch_size=batch_size,
)

In [20]:
model.to("cuda")
private_model.to("cuda")

GradSampleModule(LinearClassificationNet(
  (layer1): Linear(in_features=11, out_features=2, bias=False)
))

In [21]:
train_parameters = RegularizationConfig(
    epochs=epochs,
    device="cuda",
    batch_size=batch_size,
    seed=seed,
    optimizer="adam",
    regularization=False,
)

In [22]:
def compute_violation_with_argmax(
    sensitive_attribute_list: torch.tensor,
    analysis_dict: dict,
    y_pred: torch.tensor,
):
    return max(
        # FPR
        abs(
            (len(analysis_dict[(0, 1, 1)]) / (len(analysis_dict[(0, 1, 1)]) + len(analysis_dict[(0, 0, 1)])))
            - (len(analysis_dict[(0, 1, 0)]) / (len(analysis_dict[(0, 1, 0)]) + len(analysis_dict[(0, 0, 0)])))
        ),
        # TPR
        abs(
            (len(analysis_dict[(1, 1, 1)]) / (len(analysis_dict[(1, 1, 1)]) + len(analysis_dict[(1, 0, 1)])))
            - (len(analysis_dict[(1, 1, 0)]) / (len(analysis_dict[(1, 1, 0)]) + len(analysis_dict[(1, 0, 0)])))
        ),
    )


def compute_error_rate_difference(
    sensitive_attribute_list: torch.tensor,
    analysis_dict: dict,
    y_pred: torch.tensor,
):
    return abs(
        (
            (len(analysis_dict[(0, 1, 0)]) + len(analysis_dict[(1, 0, 0)]))
            / (
                len(analysis_dict[(0, 1, 0)])
                + len(analysis_dict[(1, 0, 0)])
                + len(analysis_dict[(1, 1, 0)])
                + len(analysis_dict[(0, 0, 0)])
            )
        )
        - (
            (len(analysis_dict[(0, 1, 1)]) + len(analysis_dict[(1, 0, 1)]))
            / (
                len(analysis_dict[(0, 1, 1)])
                + len(analysis_dict[(1, 0, 1)])
                + len(analysis_dict[(1, 1, 1)])
                + len(analysis_dict[(0, 0, 1)])
            )
        )
    )

In [23]:
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

# from .DPL.DPLUtilsutils import Utils


class MyEqualizedOddsLoss(nn.Module):
    def __init__(self, weight=None, size_average=True, estimation=0.5) -> None:
        """Initialization of the regularization loss."""
        super().__init__()
        self.estimation = estimation

    def forward(
        self,
        sensitive_attribute_list: torch.tensor,
        device: torch.device,
        predictions: torch.tensor,
        true_targets: torch.tensor,
        possible_sensitive_attributes: list,
        possible_targets: list,
        average_probabilities: dict = None,
    ) -> torch.tensor:
        fairness_violations = []
        # We compute the softmax of the predictions. We do this because
        # we can't use the argmax function on the nn output,
        # because we need differentiable results
        softmax_ = F.softmax(predictions, dim=1)

        # convert the list of sensitive attributes to a tensor and move it to the device
        sensitive_attribute_list = torch.tensor([int(item) for item in sensitive_attribute_list])
        sensitive_attribute_list = sensitive_attribute_list.to(device)
        true_targets = torch.tensor([int(item) for item in true_targets]).to(device)

        # We compute the argmax of the predictions, this is used to count
        # the number of samples for each class that are predicted with one class
        # or with the other.
        predictions_argmax = torch.argmax(torch.tensor(predictions), dim=1).to(device)
        # we convert the possible targets and the possible sensitive attributes to a list
        # just to be sure that the values are integers
        possible_targets = [int(item) for item in possible_targets]
        possible_sensitive_attributes = [int(item) for item in possible_sensitive_attributes]

        analysis_dict = {}

        sensitive_attribute_list = [1 if item == 1.0 else 0 for item in sensitive_attribute_list]

        for index, y, prediction, group in zip(
            list(range(len(predictions))),
            true_targets,
            predictions_argmax,
            sensitive_attribute_list,
        ):
            prediction = int(prediction.item())
            y = int(y.item())
            if (y, prediction, group) not in analysis_dict:
                analysis_dict[(y, prediction, group)] = []
            analysis_dict[(y, prediction, group)].append(index)

        fp_male = torch.sum(softmax_[analysis_dict[(0, 1, 1)]][:, 1])
        fp_female = torch.sum(softmax_[analysis_dict[(0, 1, 0)]][:, 1])
        tn_male = torch.sum(softmax_[analysis_dict[(0, 0, 1)]][:, 0])
        tn_female = torch.sum(softmax_[analysis_dict[(0, 0, 0)]][:, 0])

        tp_male = torch.sum(softmax_[analysis_dict[(1, 1, 1)]][:, 1])
        tp_female = torch.sum(softmax_[analysis_dict[(1, 1, 0)]][:, 1])
        fn_male = torch.sum(softmax_[analysis_dict[(1, 0, 1)]][:, 0])
        fn_female = torch.sum(softmax_[analysis_dict[(1, 0, 0)]][:, 0])

        print(
            "fp_male: ",
            fp_male,
            "fp_female: ",
            fp_female,
            "tn_male: ",
            tn_male,
            "tn_female: ",
            tn_female,
        )
        print(
            "tp_male: ",
            tp_male,
            "tp_female: ",
            tp_female,
            "fn_male: ",
            fn_male,
            "fn_female: ",
            fn_female,
        )

        fairness_violations.append(
            max(
                # FPR
                abs((fp_male / (fp_male + tn_male)) - (fp_female / (fp_female + tn_female))),
                # TPR
                abs((tp_male / (tp_male + fn_male)) - (tp_female / (tp_female + fn_female))),
            )
        )

        print(fairness_violations)

        fairness_violations_ = [item.item() if isinstance(item, torch.Tensor) else item for item in fairness_violations]

        # We get the index of the maximum violation term. Then we create a mask with
        # all zeros and we set to 1 the element at the index we found. We use this mask
        # to sum the violation terms and we return the result. This was needed because
        # when we started to work on this project we discovered that without this
        # some of the gradients were not computed correctly. I would not remove it
        # even if I'm not sure that it is needed anymore.
        index = fairness_violations_.index(max(fairness_violations_))
        fairness_violations = torch.stack(fairness_violations)
        mask = torch.full((fairness_violations.shape[0],), 0, dtype=torch.float32).to(device)
        mask[index] = 1
        res = torch.sum(mask * fairness_violations)

        return res

    def violation_with_dataset(
        self,
        model: torch.nn.Module,
        dataset: torch.utils.data.DataLoader,
        average_probabilities: dict,
        device: torch.device,
    ) -> torch.tensor:
        predictions = torch.tensor([]).to(device)
        sensitive_attribute_list = torch.tensor([]).to(device)
        targets = []
        model.eval()
        with torch.no_grad():
            for images, sensitive_attributes, target, _, _ in dataset:
                images = images.to(device)
                target = target.to(device)

                output = model(images)

                predictions = torch.cat((predictions, output), 0)
                sensitive_attribute_list = torch.cat((sensitive_attribute_list, sensitive_attributes.to(device)), 0)
                targets += target.tolist()

        sensitive_attributes = list({item.item() for item in sensitive_attribute_list})
        target_list = list(set(targets))

        # now we just call the forward function with the "fake" predictions and the sensitive
        # attribute list we computed
        return self.forward(
            sensitive_attribute_list=sensitive_attribute_list,
            device=device,
            predictions=predictions,
            true_targets=np.array(targets),
            possible_sensitive_attributes=sensitive_attributes,
            possible_targets=target_list,
            average_probabilities=average_probabilities,
        )

In [24]:
for epoch in range(0, epochs):
    # Now we can train the model. First of all we will train a model without any
    # fairness mitigation
    results = Learning.train_private_model(
        train_parameters=train_parameters,
        model=private_model,
        model_regularization=None,
        optimizer=private_optimizer,
        optimizer_regularization=None,
        train_loader=train_loader,
        test_loader=test_loader,
        average_probabilities=None,
        current_epoch=epoch,
    )
    (
        _,
        accuracy,
        _,
        _,
        _,
        max_disparity_test,
        y_true,
        y_pred,
        sensitive_attributes,
        real_indexes,
        _,
    ) = Learning.test(
        model=private_model,
        test_loader=train_loader,
        train_parameters=train_parameters,
        current_epoch=epochs,
    )
    sensitive_attributes = [1 if item == 1.0 else 0 for item in sensitive_attributes]

    analysis_dict = {}
    for index, y, prediction, group in zip(real_indexes, y_true, y_pred, sensitive_attributes):
        if (y, prediction, group) not in analysis_dict:
            analysis_dict[(y, prediction, group)] = []
        analysis_dict[(y, prediction, group)].append(index)

    for key in analysis_dict:
        print(f"{key}: {len(analysis_dict[key])}")

    fairness_module = SPATIALFairnessModule(
        _y_true=np.array(y_true),
        _y_pred=np.array(y_pred),
        _groups=np.array(sensitive_attributes),
        _group_name="Gender",
    )

    print(
        "equalized_odds_difference: ",
        fairness_module.clf_metrics.equalized_odds_difference(),
    )
    print("error_rate_difference: ", fairness_module.clf_metrics.error_rate_difference())

    print(
        "My Equalized Odds: ",
        compute_violation_with_argmax(
            sensitive_attribute_list=np.array(sensitive_attributes),
            analysis_dict=analysis_dict,
            y_pred=np.array(y_pred),
        ),
    )

    print(
        "My EO: ",
        MyEqualizedOddsLoss().violation_with_dataset(private_model, train_loader, None, "cuda"),
    )

    print(
        f"Epoch {epoch} - Train accuracy {results['Train Accuracy']} - Train Loss {results['Train Loss']} - Equalized Odds Difference {max_disparity_test}"
    )

(0, 0, 1): 11381
(0, 1, 1): 3737
(1, 1, 0): 14087
(1, 1, 1): 7441
(0, 0, 0): 6282
(1, 0, 0): 2082
(0, 1, 0): 1705
(1, 0, 1): 1621
equalized_odds_difference:  0.05011391416739375
error_rate_difference:  -0.06481544485246227
My Equalized Odds:  0.05011391416739375
fp_male:  tensor(2419.4321, device='cuda:0') fp_female:  tensor(1145.8003, device='cuda:0') tn_male:  tensor(8581.4766, device='cuda:0') tn_female:  tensor(4551.9375, device='cuda:0')
tp_male:  tensor(5448.9561, device='cuda:0') tp_female:  tensor(10340.4883, device='cuda:0') fn_male:  tensor(1067.4153, device='cuda:0') fn_female:  tensor(1376.7872, device='cuda:0')
[tensor(0.0463, device='cuda:0')]
My EO:  tensor(0.0463, device='cuda:0')
Epoch 0 - Train accuracy 0.7726539373397827 - Train Loss 0.5429727002365948 - Equalized Odds Difference 0.046304523944854736
(0, 0, 1): 11182
(1, 1, 1): 7548
(0, 0, 0): 6270
(1, 0, 0): 1983
(1, 1, 0): 14186
(0, 1, 1): 3936
(0, 1, 0): 1717
(1, 0, 1): 1514
equalized_odds_difference:  0.045377565

In [25]:
(
    _,
    accuracy,
    _,
    _,
    _,
    max_disparity_test,
    y_true_test,
    y_pred_test,
    sensitive_attributes_test,
    real_indexes_test,
    _,
) = Learning.test(
    model=private_model,
    test_loader=test_loader,
    train_parameters=train_parameters,
    current_epoch=epochs,
)

analysis_dict_test = {}
for index, y, prediction, group in zip(real_indexes_test, y_true_test, y_pred_test, sensitive_attributes_test):
    if (y, prediction, group) not in analysis_dict_test:
        analysis_dict_test[(y, prediction, group)] = []
    analysis_dict_test[(y, prediction, group)].append(index)

for key in analysis_dict_test:
    print(f"{key}: {len(analysis_dict_test[key])}")

print(f"Test accuracy {accuracy} - Equalized Odds Difference {max_disparity_test}")

fairness_module = SPATIALFairnessModule(
    _y_true=np.array(y_true_test),
    _y_pred=np.array(y_pred_test),
    _groups=np.array(sensitive_attributes_test),
    _group_name="Gender",
)

print(
    "My Equalized Odds: ",
    compute_violation_with_argmax(
        sensitive_attribute_list=np.array(sensitive_attributes_test),
        analysis_dict=analysis_dict_test,
        y_pred=np.array(y_pred_test),
    ),
)

print(
    "My Error Rate Difference: ",
    compute_error_rate_difference(
        sensitive_attribute_list=np.array(sensitive_attributes_test),
        analysis_dict=analysis_dict_test,
        y_pred=np.array(y_pred_test),
    ),
)

print(
    "equalized_odds_difference: ",
    fairness_module.clf_metrics.equalized_odds_difference(),
)
print("error_rate_difference: ", fairness_module.clf_metrics.error_rate_difference())

(1, 1, 1.0): 1839
(0, 1, 0.0): 384
(0, 0, 1.0): 2791
(1, 1, 0.0): 3695
(0, 0, 0.0): 1532
(0, 1, 1.0): 951
(1, 0, 0.0): 506
(1, 0, 1.0): 386
Test accuracy 0.8157067196292618 - Equalized Odds Difference 0.048537254333496094
My Equalized Odds:  0.0537246334281401
My Error Rate Difference:  0.078569536406423
equalized_odds_difference:  0.0537246334281401
error_rate_difference:  -0.07856953640642306


# Removing TP

In [27]:
indexes_to_remove = analysis_dict[(1, 1, 1)][0 : len(analysis_dict[(1, 1, 1)]) // 2]
indexes_to_remove_test = analysis_dict_test[(1, 1, 1)][0 : len(analysis_dict_test[(1, 1, 1)]) // 2]

In [29]:
indexes = list(set(indexes_to_remove + indexes_to_remove_test))

In [31]:
data = arff.loadarff("../dataset/dutch/dutch_census.arff")

In [38]:
from scipy.io import arff

dutch_df = pd.DataFrame(data[0]).astype("int32")

# Remove the indexes from the dataframe
dutch_df.drop(indexes, inplace=True)

dutch_df.to_csv("../dataset/dutch/dutch_census_removed.csv", index=False)

In [39]:
dutch_df.head()

,sex,age,household_position,household_size,prev_residence_place,citizenship,country_birth,edu_level,economic_status,cur_eco_activity,Marital_status,occupation
1,2,10,1122,113,1,1,1,2,111,122,2,549
2,1,8,1122,113,1,1,1,2,111,122,2,21
3,1,12,1121,112,1,1,1,1,111,137,2,549
4,2,4,1110,114,1,1,1,2,111,138,1,549
5,2,8,1122,126,1,1,1,2,120,131,2,21


In [ ]:
train_ds, test_ds = DatasetUtils.download_dataset(dataset_name="dutch", base_path="../dataset/dutch/")

In [ ]:
print(len(train_ds.samples))

In [ ]:
print(len(test_ds.samples))

In [ ]:
from FairReg.Regularization.RegularizationLoss import RegularizationLoss

RegularizationLoss().compute_violation_with_argmax(
    predictions_argmax=train_ds.targets,
    sensitive_attribute_list=train_ds.sensitive_features,
    current_target=1,
    current_sensitive_feature=1,
)

In [ ]:
# remove indexes from train_ds
train_ds.samples = np.delete(train_ds.samples, indexes_to_remove, axis=0)
train_ds.targets = np.delete(train_ds.targets, indexes_to_remove, axis=0)
train_ds.sensitive_features = np.delete(train_ds.sensitive_features, indexes_to_remove, axis=0)
train_ds.indexes = np.delete(train_ds.indexes, indexes_to_remove, axis=0)

# remove indexes from test_ds
test_ds.samples = np.delete(test_ds.samples, indexes_to_remove_test, axis=0)
test_ds.targets = np.delete(test_ds.targets, indexes_to_remove_test, axis=0)
test_ds.sensitive_features = np.delete(test_ds.sensitive_features, indexes_to_remove_test, axis=0)
test_ds.indexes = np.delete(test_ds.indexes, indexes_to_remove_test, axis=0)

In [ ]:
from FairReg.Regularization.RegularizationLoss import RegularizationLoss

RegularizationLoss().compute_violation_with_argmax(
    predictions_argmax=train_ds.targets,
    sensitive_attribute_list=train_ds.sensitive_features,
    current_target=1,
    current_sensitive_feature=1,
)

In [ ]:
torch.save(train_ds, "../dataset/dutch/train_ds_unfair.pt")
torch.save(test_ds, "../dataset/dutch/test_ds_unfair.pt")

# train_ds = torch.load("../dataset/dutch/train_ds.pt")
# test_ds = torch.load("../dataset/dutch/test_ds.pt")

In [ ]:
train_loader = torch.utils.data.DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
)

test_loader = torch.utils.data.DataLoader(
    test_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
)

In [ ]:
# Create the model that we will train, for Dutch we will use a LinearClassificationNet
# defined inside this Library in the Models/logistic_regression_net.py file
model = LinearClassificationNet()
lr = 0.019925917176300392
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
epochs = 5
seed = 42

# seed the model
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)
torch.cuda.manual_seed_all(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
os.environ["PYTHONHASHSEED"] = str(seed)

In [ ]:
# We don't want to use privacy in this case but we make the model private using
# the noise=0 because the code for learning the model only accept private models.
(
    private_model,
    private_optimizer,
    private_train_loader,
) = ModelUtils.create_private_model(
    model=model,
    epsilon=None,
    noise_multiplier=0,
    original_optimizer=optimizer,
    train_loader=train_loader,
    epochs=epochs,
    delta=0,
    MAX_GRAD_NORM=10000000000,  # since we just need to wrap the model without using privacy we use a high value here
    batch_size=batch_size,
)

In [ ]:
model.to("cuda")
private_model.to("cuda")

In [ ]:
train_parameters = RegularizationConfig(
    epochs=epochs,
    device="cuda",
    batch_size=batch_size,
    seed=seed,
    optimizer="adam",
    regularization=False,
)

In [ ]:
for epoch in range(0, epochs):
    # Now we can train the model. First of all we will train a model without any
    # fairness mitigation
    results = Learning.train_private_model(
        train_parameters=train_parameters,
        model=private_model,
        model_regularization=None,
        optimizer=private_optimizer,
        optimizer_regularization=None,
        train_loader=train_loader,
        test_loader=test_loader,
        average_probabilities=None,
        current_epoch=epoch,
    )
    (
        _,
        accuracy,
        _,
        _,
        _,
        max_disparity_test,
        y_true,
        y_pred,
        sensitive_attributes,
        real_indexes,
        _,
    ) = Learning.test(
        model=private_model,
        test_loader=train_loader,
        train_parameters=train_parameters,
        current_epoch=epochs,
    )

    analysis_dict = {}
    for index, y, prediction, group in zip(real_indexes, y_true, y_pred, sensitive_attributes):
        if (y, prediction, group) not in analysis_dict:
            analysis_dict[(y, prediction, group)] = []
        analysis_dict[(y, prediction, group)].append(index)

    for key in analysis_dict:
        print(f"{key}: {len(analysis_dict[key])}")

    fairness_module = SPATIALFairnessModule(
        _y_true=np.array(y_true),
        _y_pred=np.array(y_pred),
        _groups=np.array(sensitive_attributes),
        _group_name="Gender",
    )

    print(
        "equalized_odds_difference: ",
        fairness_module.clf_metrics.equalized_odds_difference(),
    )
    print("error_rate_difference: ", fairness_module.clf_metrics.error_rate_difference())

    print(
        f"Epoch {epoch} - Train accuracy {results['Train Accuracy']} - Train Loss {results['Train Loss']} - Equalised Odds Difference {max_disparity_test}"
    )

In [ ]:
(
    _,
    accuracy,
    _,
    _,
    _,
    max_disparity_test,
    y_true_test,
    y_pred_test,
    sensitive_attributes_test,
    real_indexes_test,
    _,
) = Learning.test(
    model=private_model,
    test_loader=test_loader,
    train_parameters=train_parameters,
    current_epoch=epochs,
)

analysis_dict_test = {}
for index, y, prediction, group in zip(real_indexes_test, y_true_test, y_pred_test, sensitive_attributes_test):
    if (y, prediction, group) not in analysis_dict_test:
        analysis_dict_test[(y, prediction, group)] = []
    analysis_dict_test[(y, prediction, group)].append(index)

for key in analysis_dict_test:
    print(f"{key}: {len(analysis_dict_test[key])}")

print(f"Test accuracy {accuracy} - Equalized Odds Difference {max_disparity_test}")

fairness_module = SPATIALFairnessModule(
    _y_true=np.array(y_true_test),
    _y_pred=np.array(y_pred_test),
    _groups=np.array(sensitive_attributes_test),
    _group_name="Gender",
)

print(
    "My Equalized Odds: ",
    compute_violation_with_argmax(
        sensitive_attribute_list=np.array(sensitive_attributes_test),
        analysis_dict=analysis_dict_test,
        y_pred=np.array(y_pred_test),
    ),
)

print(
    "equalized_odds_difference: ",
    fairness_module.clf_metrics.equalized_odds_difference(),
)
print("error_rate_difference: ", fairness_module.clf_metrics.error_rate_difference())

# Train a model with Fairness Mitigation

In [ ]:
batch_size = 333
# torch.save(train_ds, "../dataset/dutch/train_ds.pt")
# torch.save(test_ds, "../dataset/dutch/test_ds.pt")

train_ds = torch.load("../dataset/dutch/train_ds.pt")
test_ds = torch.load("../dataset/dutch/test_ds.pt")

In [ ]:
len(train_ds.samples)

In [ ]:
# sample 8000 samples from the train_ds.samples
indexes = np.random.choice(train_ds.samples.shape[0], 8000, replace=False)

In [ ]:
from utils.dutch import TabularDataset

samples = train_ds.samples[indexes]
targets = train_ds.targets[indexes]
sensitive_features = train_ds.sensitive_features[indexes]

validation_ds = TabularDataset(samples, targets, sensitive_features)

In [ ]:
train_ds.samples = np.delete(train_ds.samples, indexes, axis=0)
train_ds.targets = np.delete(train_ds.targets, indexes, axis=0)
train_ds.sensitive_features = np.delete(train_ds.sensitive_features, indexes, axis=0)
train_ds.indexes = np.delete(train_ds.indexes, indexes, axis=0)

In [ ]:
len(train_ds.samples)

In [ ]:
validation_ds

In [ ]:
train_loader = torch.utils.data.DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
)

test_loader = torch.utils.data.DataLoader(
    test_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
)

In [ ]:
# Create the model that we will train, for Dutch we will use a LinearClassificationNet
# defined inside this Library in the Models/logistic_regression_net.py file
model = LinearClassificationNet()
model_regularization = LinearClassificationNet()

lr = 0.019925917176300392
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
optimizer_regularization = torch.optim.SGD(model_regularization.parameters(), lr=lr)

epochs = 40
seed = 42

# seed the model
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)
torch.cuda.manual_seed_all(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
os.environ["PYTHONHASHSEED"] = str(seed)

In [ ]:
# We don't want to use privacy in this case but we make the model private using
# the noise=0 because the code for learning the model only accept private models.
(
    private_model,
    private_optimizer,
    private_train_loader,
) = ModelUtils.create_private_model(
    model=model,
    epsilon=None,
    noise_multiplier=0,
    original_optimizer=optimizer,
    train_loader=train_loader,
    epochs=epochs,
    delta=0,
    MAX_GRAD_NORM=10000000000,  # since we just need to wrap the model without using privacy we use a high value here
    batch_size=batch_size,
)

In [ ]:
# We don't want to use privacy in this case but we make the model private using
# the noise=0 because the code for learning the model only accept private models.

(
    private_model_regularization,
    private_optimizer_regularization,
    _,
) = ModelUtils.create_private_model(
    model=model_regularization,
    epsilon=None,
    noise_multiplier=0,
    original_optimizer=optimizer_regularization,
    train_loader=train_loader,
    epochs=epochs,
    delta=0,
    MAX_GRAD_NORM=10000000000,  # since we just need to wrap the model without using privacy we use a high value here
    batch_size=batch_size,
)

In [ ]:
model.to("cuda")
private_model.to("cuda")

In [ ]:
train_parameters = RegularizationConfig(
    epochs=epochs,
    device="cuda",
    batch_size=batch_size,
    seed=seed,
    regularization=True,
    target=0.05,
    regularization_mode="fixed",
    regularization_lambda=0.9,
    optimizer="adam",
)

In [ ]:
for epoch in range(0, epochs):
    # Now we can train the model. First of all we will train a model without any
    # fairness mitigation
    results = Learning.train_private_model(
        train_parameters=train_parameters,
        model=private_model,
        model_regularization=private_model_regularization,
        optimizer=private_optimizer,
        optimizer_regularization=private_optimizer_regularization,
        train_loader=train_loader,
        test_loader=test_loader,
        average_probabilities=None,
        current_epoch=epoch,
    )
    (
        _,
        accuracy,
        _,
        _,
        _,
        max_disparity_test,
        y_true,
        y_pred,
        sensitive_attributes,
        _,
        _,
    ) = Learning.test(
        model=private_model,
        test_loader=train_loader,
        train_parameters=train_parameters,
        current_epoch=epochs,
    )

    fairness_module = SPATIALFairnessModule(
        _y_true=np.array(y_true),
        _y_pred=np.array(y_pred),
        _groups=np.array(sensitive_attributes),
        _group_name="Gender",
    )

    print(
        "equalized_odds_difference: ",
        fairness_module.clf_metrics.equalized_odds_difference(),
    )
    print("error_rate_difference: ", fairness_module.clf_metrics.error_rate_difference())

    print(
        f"Epoch {epoch} - Train accuracy {results['Train Accuracy']} - Train Loss {results['Train Loss']} - Equalized Odds Difference {max_disparity_test} - Regularization Loss {results['Train Loss + Regularizaion']}"
    )

In [ ]:
# We can test the trained model on the test dataset to understand if the model is less unfair than before
(
    _,
    accuracy,
    _,
    _,
    _,
    _,
    y_true_test,
    y_pred_test,
    sensitive_attribute_test,
) = Learning.test(
    model=private_model,
    test_loader=test_loader,
    train_parameters=train_parameters,
    current_epoch=epochs,
)

_, _, _, _, _, max_disparity_test, _, _, _ = Learning.test(
    model=private_model,
    test_loader=test_loader,
    train_parameters=train_parameters,
    current_epoch=epochs,
)

print(f"Test accuracy {accuracy} - Average Predictive Value Difference  {max_disparity_test}")

fairness_module = SPATIALFairnessModule(
    _y_true=np.array(y_true_test),
    _y_pred=np.array(y_pred_test),
    _groups=np.array(sensitive_attribute_test),
    _group_name="Gender",
)

print(
    "equalized_odds_difference: ",
    fairness_module.clf_metrics.equalized_odds_difference(),
)
print("error_rate_difference: ", fairness_module.clf_metrics.error_rate_difference())